# Exercise: Sentiment Analysis on Real Movie Reviews
## AIAT 121 - Natural Language Processing

## Learning Objectives

- Preprocess real review text that has already been through somebody else's pipeline
- Build, train and evaluate a sentiment classifier on a held-out set you touch **once**
- Read what the model learned, find where it fails, and say so in writing

## 🔗 Where this fits

**Builds on:** Unit 3 example `01_text_classification.ipynb` — the same data source and the same two
classifiers, now with your own choices at every step. **Used later in:** Unit 4 lesson 03, where a pretrained
transformer is scored on reviews from the same collection, and Unit 5, whose bias audit inspects classifiers
built this way.

## Real-World Context

A streaming service wants to know, from the text alone, whether a viewer liked a film. The labelled reviews
come from the **IMDB Large Movie Review Dataset** (Maas et al., ACL 2011): 25,000 training and 25,000 test
reviews, *positive* meaning the author rated the film 7-10 and *negative* meaning 1-4. The data cell below
draws 2,000 training reviews and 1,000 test reviews with a fixed seed — a **different** draw from the
example notebook's, on purpose, so its numbers are not your numbers.

**Task**: classify reviews as positive or negative — and say honestly how well it works.

---

## Task 1: Data Preprocessing (25 points)

Preprocess the training reviews:
1. Clean the text — the data cell prints what somebody else's pipeline left behind; decide what to do about it
2. Tokenize
3. Remove stop words — and decide, in a comment, what happens to *not*, *no* and *never* (the example
   notebook measured how many reviews contain them)
4. Create TF-IDF features, **fitted on the training reviews only**


## 📥 Inputs & 📤 Outputs

**Inputs:** 2,000 labelled training reviews and 1,000 labelled test reviews from the IMDB dataset, decoded
from the Keras dataset cache by the data cell below (the two files, about 19 MB, download once if the cache
is empty). scikit-learn and NumPy; NLTK if you choose to use its tokenizer.

**Outputs:** whatever your completed TODO cells print — the before/after of your preprocessing, accuracy and a
classification report on the test set next to the majority-class baseline, and the evidence for your written
analysis. The TODO cells produce no output until you complete them.

---


## The Dataset

Run this cell first. Every task below works on `train_texts` / `train_labels`. The test set,
`test_texts` / `test_labels`, is for the **end of Task 2 only** — a test set you look at while building the
model is no longer a test set.


In [1]:
# Dataset cell: 2,000 labelled training reviews and 1,000 test reviews, drawn from the IMDB
# Large Movie Review Dataset with a fixed seed. Real text, exactly as delivered: read the
# honesty lines it prints before you touch anything.

import json
import urllib.request
import warnings
from pathlib import Path

import numpy as np

# The IMDB reviews, read straight from the Keras cache with NumPy: no TensorFlow needed.
KERAS_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/"
cache = Path.home() / ".keras" / "datasets"          # where Keras itself keeps these files
cache.mkdir(parents=True, exist_ok=True)
for fname in ("imdb.npz", "imdb_word_index.json"):
    if not (cache / fname).exists():                 # first run on this machine only
        print(f"Downloading {fname} into {cache} (once) ...")
        urllib.request.urlretrieve(KERAS_URL + fname, cache / fname)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # the object arrays pickled inside imdb.npz trip a NumPy 2.4 deprecation notice
    with np.load(cache / "imdb.npz", allow_pickle=True) as npz:
        imdb = {k: npz[k] for k in ("x_train", "y_train", "x_test", "y_test")}
id_to_word = {i: w for w, i in json.load(open(cache / "imdb_word_index.json")).items()}


def decode(ids):
    return " ".join(id_to_word.get(int(i), "?") for i in ids)


rng = np.random.default_rng(7)                       # seed 7: a different draw from the example notebook's seed 0
i_train = rng.choice(len(imdb["x_train"]), 2000, replace=False)
i_test = rng.choice(len(imdb["x_test"]), 1000, replace=False)
train_texts = [decode(r) for r in imdb["x_train"][i_train]]
train_labels = ["positive" if y == 1 else "negative" for y in imdb["y_train"][i_train]]
test_texts = [decode(r) for r in imdb["x_test"][i_test]]
test_labels = ["positive" if y == 1 else "negative" for y in imdb["y_test"][i_test]]

print(f"Training reviews: {len(train_texts):,} "
      f"({train_labels.count('positive')} positive / {train_labels.count('negative')} negative)")
print(f"Test reviews:     {len(test_texts):,} "
      f"({test_labels.count('positive')} positive / {test_labels.count('negative')} negative)"
      "   <- touch these ONCE, at the end of Task 2")
majority = max(test_labels.count("positive"), test_labels.count("negative")) / len(test_labels)
print(f"Majority-class baseline on the test set: {majority:.1%}  <- the number your model must beat")

# What somebody else's preprocessing left behind — counted, not assumed.
print(f"\nMedian words per training review: {int(np.median([len(t.split()) for t in train_texts]))}")
print(f"Training reviews containing the stray token 'br' (an HTML line break): "
      f"{np.mean([' br ' in t for t in train_texts]):.0%}")
print(f"Training reviews containing any of . , ! ? : "
      f"{np.mean([any(c in t for c in '.,!?') for t in train_texts]):.0%}")
print(f"Training reviews containing the word 'not': {np.mean([' not ' in f' {t} ' for t in train_texts]):.0%}")
print(f"\nOne {train_labels[0]} review, as delivered:\n  {train_texts[0][:400]} ...")


Training reviews: 2,000 (983 positive / 1017 negative)
Test reviews:     1,000 (504 positive / 496 negative)   <- touch these ONCE, at the end of Task 2
Majority-class baseline on the test set: 50.4%  <- the number your model must beat

Median words per training review: 175
Training reviews containing the stray token 'br' (an HTML line break): 59%
Training reviews containing any of . , ! ? : 0%
Training reviews containing the word 'not': 59%

One positive review, as delivered:
  if you haven't seen this you do not know what you are missing the first time you do you will litteraly be in pain lying on floor throwing up from laughing so hard and having probably wet yourself as well br br it is that funny there hasn't been a single comedic performance to this date that i have seen that tops this or even comes close so many classic one liners stories and segways br br the drun ...


In [2]:
# Task 1 scaffold: implement the preprocessing pipeline from Unit 1, adapted to what the
# data cell showed you. It must return one cleaned STRING per review (scikit-learn's
# TfidfVectorizer takes raw text and does its own splitting).

from sklearn.feature_extraction.text import TfidfVectorizer

STOP_WORDS = {
    # TODO: your stop-word set. Write one comment line saying what you did with negations and why.
}


def preprocess_reviews(reviews):
    """Lowercase, handle the 'br' artefact, tokenize, drop stop words, re-join."""
    # YOUR CODE HERE
    pass


# TODO: apply it, print three reviews before/after, and print the vocabulary size before/after.
# YOUR CODE HERE


## Task 2: Model Training (30 points)

Train and evaluate:
1. Vectorize with TF-IDF — `fit_transform` on the training reviews, `transform` on the test reviews. Never
   the other way round.
2. Train **two** classifiers on the same features: Multinomial Naive Bayes and Logistic Regression.
3. Evaluate both on the test set, **once**: accuracy against the majority baseline the data cell printed,
   plus a `classification_report`.
4. Print which model won and by how many reviews out of 1,000.


In [3]:
# Task 2 scaffold: TF-IDF features, two classifiers, one honest evaluation on the test set.
# Keep the test reviews unseen until the final scoring — evaluating on data the model has
# already memorized always looks deceptively good.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB

# TODO: vectorize (fit on training reviews only), train both models, score the test set once.
# YOUR CODE HERE


## Task 3: Real-World Analysis (45 points)

Analyze what you built and generate insights:
1. Which words does your better classifier associate most strongly with each class? (Logistic Regression:
   `coef_`; Naive Bayes: `feature_log_prob_`.) Print the top 10 per class — and flag any that are not
   sentiment words.
2. Print the five test reviews your model got wrong **with the highest confidence**
   (`predict_proba`), with their true labels. Read them. Why might each one have fooled a bag-of-words model?
3. Summarize your findings in 3-5 sentences in a markdown cell. One of the sentences must name a limitation
   you *measured* in this notebook, not one you remember from the lesson.

**Submission**: Complete notebook with model and analysis.

**Grading**: 100 points total


In [4]:
# TODO (Task 3): analysis code — inspect feature weights, find the most confident
# mistakes, and print the evidence for your written summary.

# YOUR CODE HERE


## 💬 Discuss

1. Your two models disagree on some reviews. Before looking at the labels, which of the two would you
   trust on a review they disagree about, and what printed number is your reason?
2. The data cell printed how many reviews contain "not". If you kept negation words in Task 1, what did
   it cost you in vocabulary size — and did the classifier actually use them? (Check the weight of "not".)


## ⚠️ Where this breaks

**The dataset has no middle.** Maas et al. built it from reviews rated 7-10 and 1-4 only; ratings of 5 and 6
were left out. A model trained here has never seen an "it was fine" review — the most common review in any
real inbox — and will call every one of them positive or negative with full confidence. Before you deploy a
sentiment model, ask what the labelled data *excluded*.

**The preprocessing was not yours.** The Keras copy is lowercased, has no punctuation, and carries `br`
tokens from HTML. Nothing you do in Task 1 can recover an exclamation mark or a capitalised SHOUT, both of
which carry sentiment. When you are handed text at work, the first job is to find out what was already done
to it.

**English only.** The stop-word decisions you made in Task 1 do not transfer to Arabic, where negation is a
prefix or a separate particle and the stop-word lists are far less mature. Unit 2's tokenization exercise is
where that problem starts.
